In [9]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
from PIL import Image
import timm

# --- CONFIGURATION ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')

# Pushing Resolution to 384 for Stage 4 detection
IMG_SIZE = 384 
BATCH_SIZE = 16 # Lowered for memory at 384px
LR = 3e-5 
EPOCHS = 15

# --- NOVELTY 1: FOCAL LOSS ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', label_smoothing=0.1)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        return focal_loss.mean()

# --- NOVELTY 2: ENHANCED ARCHITECTURE ---
class FinalAptosNet(nn.Module):
    def __init__(self):
        super().__init__()
        # ConvNeXt-Tiny is highly robust at higher resolutions
        self.backbone = timm.create_model('convnext_tiny', pretrained=True, num_classes=0)
        dim = 768
        self.attn = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(dim),
            nn.Dropout(0.3),
            nn.Linear(dim, 5)
        )

    def forward(self, x):
        feat = self.backbone(x)
        # Saliency gating
        refined = feat * self.attn(feat)
        return self.classifier(refined)

# --- DATASET ---
class AptosDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# --- ENGINE ---
def run_master_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FinalAptosNet().to(device)

    # Sampling for Clinical Relevance
    train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_1.csv'))
    counts = train_df.diagnosis.value_counts()
    cw = 1. / counts
    cw[3] *= 3.0 # Severe focus
    cw[4] *= 2.0 # Proliferative focus
    sample_weights = cw[train_df.diagnosis].values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), batch_size=BATCH_SIZE)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    criterion = FocalLoss(gamma=2.5) # Gamma > 2.0 increases focus on hard classes
    
    best_kappa = 0
    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                v_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
                v_labels.extend(labels.cpu().numpy())
        
        kappa = cohen_kappa_score(v_labels, v_preds, weights='quadratic')
        print(f"Epoch {epoch+1} | Kappa: {kappa:.4f}")
        
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(model.state_dict(), 'master_aptos_model.pth')

    # --- NOVELTY 3: TEST TIME AUGMENTATION (TTA) ---
    print("\nStarting Final TTA Evaluation...")
    model.load_state_dict(torch.load('master_aptos_model.pth'))
    model.eval()
    
    tta_preds, final_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            # Predict on 3 versions and average
            out1 = model(imgs)
            out2 = model(torch.flip(imgs, [3])) # Horizontal
            out3 = model(torch.flip(imgs, [2])) # Vertical
            
            avg_out = (out1 + out2 + out3) / 3
            tta_preds.extend(torch.argmax(avg_out, 1).cpu().numpy())
            final_labels.extend(labels.numpy())

    print("\n" + "="*50)
    print("      FINAL CLINICAL PERFORMANCE (WITH TTA)")
    print("="*50)
    print(f"Testing Accuracy: {accuracy_score(final_labels, tta_preds):.4%}")
    print(f"Quadratic Kappa:  {cohen_kappa_score(final_labels, tta_preds, weights='quadratic'):.4f}")
    print("\nClassification Report:\n", classification_report(final_labels, tta_preds))

if __name__ == "__main__":
    run_master_training()

Epoch 1 | Kappa: 0.7114
Epoch 2 | Kappa: 0.6782
Epoch 3 | Kappa: 0.6785
Epoch 4 | Kappa: 0.8852
Epoch 5 | Kappa: 0.8274
Epoch 6 | Kappa: 0.8580
Epoch 7 | Kappa: 0.8738
Epoch 8 | Kappa: 0.9013
Epoch 9 | Kappa: 0.8535
Epoch 10 | Kappa: 0.8791
Epoch 11 | Kappa: 0.8513
Epoch 12 | Kappa: 0.8540
Epoch 13 | Kappa: 0.7898
Epoch 14 | Kappa: 0.8514
Epoch 15 | Kappa: 0.8664

Starting Final TTA Evaluation...

      FINAL CLINICAL PERFORMANCE (WITH TTA)
Testing Accuracy: 79.5082%
Quadratic Kappa:  0.8941

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99       172
           1       0.63      0.80      0.70        40
           2       0.87      0.57      0.69       104
           3       0.33      0.77      0.46        22
           4       0.56      0.54      0.55        28

    accuracy                           0.80       366
   macro avg       0.68      0.73      0.68       366
weighted avg       0.85      0.80      0.81 